# Lab 5: Re-ranking for Better Retrieval

**Level:** Basic | **Duration:** ~35 minutes

## What You'll Learn
- Why initial retrieval (bi-encoder) often returns suboptimal results
- How cross-encoder re-ranking improves precision
- The retrieve-then-rerank pattern: cast a wide net, then refine
- How to measure the impact of re-ranking on answer quality
- The latency vs. accuracy trade-off

## The Problem
Bi-encoders (like bi-encoder embeddings) encode query and document independently. They're fast but miss nuanced query-document interactions. Cross-encoders process query and document together, capturing deeper relevance signals — but they're slower.

## Setup

In [ ]:
!pip install -q langchain langchain-groq langchain-community sentence-transformers langchain-chroma chromadb sentence-transformers

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

import time
import numpy as np
from langchain_groq import ChatGroq, HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

# Load cross-encoder for re-ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Setup complete! Cross-encoder loaded.")

## Step 1: Build a Larger Knowledge Base

Re-ranking shines when there's more content to sift through. We need enough documents that initial retrieval returns a mix of relevant and semi-relevant results.

In [ ]:
knowledge_base = """
# SkyWing Airlines - Comprehensive Operations Guide

## Booking and Ticketing

SkyWing offers three fare classes: Light, Flex, and Premium. Light fares are the most affordable but come with restrictions - no free checked baggage, no free seat selection, and no refunds. Flex fares include one checked bag, free seat selection, and free rebooking. Premium fares include two checked bags, priority boarding, lounge access, and full flexibility.

Group bookings of 10 or more passengers receive a 15% discount on Flex and Premium fares. Group bookings must be made at least 30 days before departure through the dedicated group desk. Payment for group bookings is due 21 days before departure.

Corporate accounts can access negotiated rates through the SkyWing Business Portal. Corporate fares typically offer 10-25% savings compared to published fares, with full flexibility included. Monthly billing and centralized reporting are available for corporate accounts.

## Check-in Procedures

Online check-in opens 24 hours before departure and closes 1 hour before departure for domestic flights, 2 hours for international flights. Mobile boarding passes are accepted at all SkyWing-operated airports. Passengers checking in at the airport must arrive at least 2 hours before domestic flights and 3 hours before international flights.

Self-service kiosks are available at major airports for baggage drop and boarding pass printing. Kiosks support 12 languages and can process special requests such as meal preferences and seat changes. Staff-assisted check-in is available for passengers requiring special assistance, unaccompanied minors, and passengers traveling with pets.

## Baggage

Carry-on allowance: all passengers may bring one cabin bag (max 55x40x20cm, 8kg) and one personal item (max 40x30x15cm). Checked baggage: Light fare passengers must purchase checked baggage separately (from 15 euros per bag). Flex passengers receive one free checked bag up to 23kg. Premium passengers receive two free checked bags up to 23kg each.

Oversized items such as sports equipment, musical instruments, and surfboards are subject to special handling fees ranging from 30 to 75 euros depending on the item. Fragile items can be transported with a fragile handling tag for an additional 10 euros. SkyWing is not liable for damage to items not properly packaged.

Lost baggage must be reported at the airport baggage service desk immediately upon arrival. SkyWing will deliver recovered bags to the passenger's address within 24 hours for domestic flights and 48 hours for international flights. If baggage is not found within 21 days, compensation is provided per the Montreal Convention.

## In-flight Services

All SkyWing flights offer complimentary water and a snack on flights over 1 hour. On flights over 3 hours, a full meal service is provided for Flex and Premium passengers. Light fare passengers can purchase meals and beverages from the buy-on-board menu, with prices ranging from 3 to 12 euros.

Wi-Fi is available on all long-haul flights and 60% of short-haul aircraft. Pricing is 5 euros for 1 hour, 10 euros for 3 hours, or 15 euros for the full flight. Premium passengers receive complimentary Wi-Fi. Streaming entertainment is available on personal devices through the SkyWing app on all Wi-Fi equipped aircraft.

## Loyalty Program - SkyWing Miles

Members earn 1 mile per kilometer flown on Light fares, 1.5 miles on Flex, and 2 miles on Premium. Bonus miles are earned based on tier status: Silver (25% bonus), Gold (50% bonus), Platinum (100% bonus). Miles expire after 24 months of account inactivity.

Tier qualification requires minimum flight activity in a calendar year: Silver (25,000 miles or 30 segments), Gold (50,000 miles or 60 segments), Platinum (100,000 miles or 120 segments). Tier benefits include priority check-in, extra baggage, lounge access (Gold+), and upgrade priority.

Miles can be redeemed for flights starting at 10,000 miles for short-haul and 30,000 miles for long-haul. Partner redemptions include hotel stays, car rentals, and retail purchases. Miles can be transferred between family members in the same household at no cost.

## Safety and Compliance

SkyWing holds IOSA certification and undergoes annual safety audits. All aircraft are maintained according to EASA Part-145 standards. Safety Management System (SMS) reports are filed through the confidential reporting system. Employees are encouraged to report safety concerns without fear of retribution.

Cabin crew must complete 4 weeks of initial safety training and annual recurrent training. Training covers emergency procedures, first aid, dangerous goods handling, and CRM (Crew Resource Management). All cabin crew are trained to use all emergency equipment including defibrillators, oxygen systems, and fire suppression equipment.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_text(knowledge_base)
docs = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(chunks)]

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="reranking_test",
)

print(f"Indexed {len(chunks)} chunks (small chunks to make retrieval harder).")

## Step 2: Retrieve a Wide Set (Top 20)

The first stage retrieval casts a wide net. We deliberately over-retrieve, knowing that many results will be only partially relevant.

In [ ]:
def initial_retrieve(query, k=20):
    """First stage: bi-encoder retrieval with a wide net."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)

query = "What are the benefits for frequent flyers?"
initial_results = initial_retrieve(query)

print(f"Query: \"{query}\"")
print(f"Retrieved {len(initial_results)} chunks:\n")
for i, doc in enumerate(initial_results, 1):
    print(f"{i:>2}. {doc.page_content[:90]}...")

**Notice:** Some of these chunks are highly relevant (loyalty program benefits), but others are only tangentially related (check-in, baggage). The bi-encoder can't distinguish well between these.

## Step 3: Re-rank with Cross-Encoder

The cross-encoder takes the query and each candidate document as a pair and scores their relevance directly. This is much more accurate but slower since each pair requires a forward pass.

In [ ]:
def rerank(query, documents, top_k=5):
    """Re-rank documents using a cross-encoder."""
    # Create query-document pairs
    pairs = [[query, doc.page_content] for doc in documents]

    # Score all pairs
    scores = cross_encoder.predict(pairs)

    # Sort by score (descending)
    scored_docs = list(zip(scores, documents))
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    return scored_docs[:top_k]

# Re-rank the initial results
reranked = rerank(query, initial_results, top_k=5)

print(f"Top 5 after re-ranking:\n")
for i, (score, doc) in enumerate(reranked, 1):
    print(f"{i}. [score: {score:.4f}] {doc.page_content[:100]}...")
    print()

## Step 4: Compare Retrieval Quality

Let's see the difference between top-5 from bi-encoder alone vs. top-5 after re-ranking.

In [ ]:
query = "What are the benefits for frequent flyers?"

# Bi-encoder only (top 5)
biencoder_top5 = initial_retrieve(query, k=5)

# Retrieve top 20, then re-rank to top 5
initial_top20 = initial_retrieve(query, k=20)
reranked_top5 = rerank(query, initial_top20, top_k=5)

print("TOP 5: Bi-encoder only")
print("-" * 60)
for i, doc in enumerate(biencoder_top5, 1):
    print(f"  {i}. {doc.page_content[:100]}...")

print(f"\nTOP 5: After re-ranking (from top 20)")
print("-" * 60)
for i, (score, doc) in enumerate(reranked_top5, 1):
    print(f"  {i}. [score: {score:.4f}] {doc.page_content[:100]}...")

## Step 5: Impact on Answer Quality

Now let's build two RAG pipelines and compare their answer quality.

In [ ]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
Be specific and cite details from the context.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def rag_without_reranking(question, k=5):
    """Standard RAG: retrieve top-k, generate."""
    retrieved = initial_retrieve(question, k=k)
    context = format_docs(retrieved)
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer

def rag_with_reranking(question, initial_k=20, final_k=5):
    """Re-ranking RAG: retrieve top-20, re-rank to top-5, generate."""
    initial = initial_retrieve(question, k=initial_k)
    reranked = rerank(question, initial, top_k=final_k)
    docs = [doc for _, doc in reranked]
    context = format_docs(docs)
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer

# Compare on several questions
test_questions = [
    "What are the benefits for frequent flyers?",
    "How does the baggage compensation process work?",
    "What training do cabin crew receive?",
    "How do corporate accounts get discounts?",
    "What Wi-Fi options are available on flights?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    print("=" * 70)

    answer_no_rr = rag_without_reranking(q)
    answer_with_rr = rag_with_reranking(q)

    print(f"\n[Without Re-ranking]")
    print(f"{answer_no_rr[:250]}")
    print(f"\n[With Re-ranking]")
    print(f"{answer_with_rr[:250]}")
    print()

## Step 6: Measure the Latency Trade-off

Re-ranking adds compute. Let's quantify it.

In [ ]:
question = "What are the benefits for frequent flyers?"

# Time bi-encoder retrieval only
start = time.time()
for _ in range(5):
    initial_retrieve(question, k=5)
biencoder_time = (time.time() - start) / 5

# Time retrieval + re-ranking
start = time.time()
for _ in range(5):
    initial = initial_retrieve(question, k=20)
    rerank(question, initial, top_k=5)
rerank_time = (time.time() - start) / 5

# Time full RAG without re-ranking
start = time.time()
for _ in range(3):
    rag_without_reranking(question)
full_no_rr = (time.time() - start) / 3

# Time full RAG with re-ranking
start = time.time()
for _ in range(3):
    rag_with_reranking(question)
full_with_rr = (time.time() - start) / 3

print("Latency Comparison (averaged)")
print("=" * 50)
print(f"{'Stage':<35} {'Time':>10}")
print("-" * 50)
print(f"{'Bi-encoder retrieval (top 5)':<35} {biencoder_time*1000:>8.0f} ms")
print(f"{'Retrieve 20 + re-rank to 5':<35} {rerank_time*1000:>8.0f} ms")
print(f"{'Re-ranking overhead':<35} {(rerank_time-biencoder_time)*1000:>8.0f} ms")
print(f"{'Full RAG (no re-ranking)':<35} {full_no_rr*1000:>8.0f} ms")
print(f"{'Full RAG (with re-ranking)':<35} {full_with_rr*1000:>8.0f} ms")
print(f"\nRe-ranking adds ~{(rerank_time-biencoder_time)*1000:.0f}ms to retrieval.")
print(f"In a full RAG pipeline, this is often negligible vs. the LLM generation time.")

## Step 7: Tuning the Retrieve-Then-Rerank Pipeline

How many initial candidates should you retrieve before re-ranking? Let's experiment.

In [ ]:
question = "How does the baggage compensation process work?"

print(f"Q: {question}\n")

for initial_k in [5, 10, 20, 30]:
    start = time.time()
    initial = initial_retrieve(question, k=initial_k)
    reranked = rerank(question, initial, top_k=3)
    elapsed = time.time() - start

    top_score = reranked[0][0]
    top_text = reranked[0][1].page_content[:80]

    print(f"initial_k={initial_k:>2}: top score={top_score:.4f}, time={elapsed*1000:.0f}ms")
    print(f"  Top chunk: {top_text}...")
    print()

print("Generally, initial_k=10-20 gives the best quality/speed balance.")

## Step 8: Visualizing Score Distributions

Let's see how the re-ranker reshuffles the initial retrieval order.

In [ ]:
import matplotlib.pyplot as plt

question = "What training do cabin crew receive?"
initial = initial_retrieve(question, k=15)

# Get cross-encoder scores
pairs = [[question, doc.page_content] for doc in initial]
ce_scores = cross_encoder.predict(pairs)

# Plot original rank vs cross-encoder score
fig, ax = plt.subplots(figsize=(10, 5))
original_ranks = range(1, len(ce_scores) + 1)

bars = ax.bar(original_ranks, ce_scores, color=["#2ecc71" if s > 0 else "#e74c3c" for s in ce_scores])
ax.set_xlabel("Original bi-encoder rank")
ax.set_ylabel("Cross-encoder relevance score")
ax.set_title(f"Re-ranking: '{question}'")
ax.axhline(y=0, color="black", linestyle="--", alpha=0.3)

# Mark the top 3 after re-ranking
reranked_indices = sorted(range(len(ce_scores)), key=lambda i: ce_scores[i], reverse=True)[:3]
for idx in reranked_indices:
    bars[idx].set_edgecolor("black")
    bars[idx].set_linewidth(2)
    ax.annotate(f"Top {reranked_indices.index(idx)+1}", (idx+1, ce_scores[idx]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print("Green = positive relevance, Red = negative relevance")
print("Black-bordered bars = top 3 after re-ranking")
print("\nNote how re-ranking often promotes documents that the bi-encoder ranked lower.")

---

## YOUR TURN

Try a different cross-encoder model and compare its re-ranking quality.

Options to try:
- `cross-encoder/ms-marco-MiniLM-L-12-v2` (larger, potentially more accurate)
- `cross-encoder/stsb-roberta-base` (trained on semantic similarity)
- `BAAI/bge-reranker-base` (multilingual reranker)

Compare the top-5 results from each model on the same queries.

In [ ]:
# YOUR TURN: Try a different cross-encoder

# model_name = "cross-encoder/ms-marco-MiniLM-L-12-v2"  # Uncomment to try
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # Current model

alt_cross_encoder = CrossEncoder(model_name)

question = "What are the benefits for frequent flyers?"
initial = initial_retrieve(question, k=20)

# Score with both models
pairs = [[question, doc.page_content] for doc in initial]
scores_original = cross_encoder.predict(pairs)
scores_alt = alt_cross_encoder.predict(pairs)

# Compare top 5
top5_original = sorted(range(len(scores_original)), key=lambda i: scores_original[i], reverse=True)[:5]
top5_alt = sorted(range(len(scores_alt)), key=lambda i: scores_alt[i], reverse=True)[:5]

print("Original model top 5:")
for rank, idx in enumerate(top5_original, 1):
    print(f"  {rank}. [score: {scores_original[idx]:.4f}] {initial[idx].page_content[:80]}...")

print(f"\nAlternative model ({model_name}) top 5:")
for rank, idx in enumerate(top5_alt, 1):
    print(f"  {rank}. [score: {scores_alt[idx]:.4f}] {initial[idx].page_content[:80]}...")

overlap = set(top5_original) & set(top5_alt)
print(f"\nOverlap: {len(overlap)} of 5 chunks in common.")

## Key Takeaways

1. **Bi-encoders are fast but imprecise** — they encode query and document separately.
2. **Cross-encoders are slow but precise** — they see query and document together.
3. **Retrieve-then-rerank** combines the best of both: wide initial retrieval, precise final selection.
4. **Typical pattern:** Retrieve top 20 with bi-encoder, re-rank to top 5 with cross-encoder.
5. **The latency overhead** of re-ranking is usually small compared to LLM generation time.
6. **Model choice matters** — different cross-encoders have different strengths.

**Next:** In Lab 6, we'll build an agentic RAG system that decides when and how to retrieve.